# Descriptive Statistics: Activity: Steps Features

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')  # go up to scripts/ root

import pandas as pd
from datetime import datetime
import numpy as np
from scipy import stats

import matplotlib.pyplot as plt





# Functions

In [ ]:
def check_normality(data, feature_name="Feature", ax=None):
    """
    Generate Q-Q plot and run Shapiro-Wilk test for a given array/series.
    
    Parameters
    ----------
    data : array-like
        Numeric data to test (NaNs are dropped automatically).
    feature_name : str
        Label used in the plot title and printed output.
    ax : matplotlib axis, optional
        If provided, plots on this axis (useful for subplots/loops).
        Otherwise creates its own figure.
    
    Returns
    -------
    dict with W statistic, p-value, and a boolean for normality at alpha=0.05
    """
    data = pd.Series(data).dropna().values

    # Shapiro-Wilk test
    W, p_value = stats.shapiro(data)
    is_normal = p_value > 0.05

    # Q-Q plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {feature_name}\nShapiro-Wilk W={W:.3f}, p={p_value:.4f}")

    print(f"{feature_name}: W={W:.4f}, p={p_value:.4f} -> "
          f"{'Normal (fail to reject H0)' if is_normal else 'Non-normal (reject H0)'}")

    return {"feature": feature_name, "W": W, "p_value": p_value, "is_normal": is_normal}



# --- Example: loop over multiple features in a feature matrix ---
def check_normality_batch(df, feature_cols, ncols=4):
    n = len(feature_cols)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    results = []
    for i, col in enumerate(feature_cols):
        data = df[col].dropna()
        W, p_value = stats.shapiro(data)
        skewness = stats.skew(data, bias=False)

        if abs(skewness) < 0.5:
            direction = "symmetric"
        elif skewness >= 0.5:
            direction = "right-skewed"
        else:
            direction = "left-skewed"

        stats.probplot(data, dist="norm", plot=axes[i])
        axes[i].set_title(f"{col}\nW={W:.3f}, p={p_value:.4f}\nskew={skewness:.2f} ({direction})")

        results.append({
            "feature": col, "W": W, "p_value": p_value,
            "is_normal": p_value > 0.05,
            "skewness": skewness, "direction": direction
        })

    for j in range(len(feature_cols), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(results)

## Calculate descriptive statistics

In [ ]:
df_step_features = pd.read_csv('../../output/1_feature_extraction/df_features_step_2026-07-08.csv')
#!exclude DEC_46 from all step features
df_step = df_step_features[df_step_features['study_id'] != 'DEC_46']

df_stats = df_step_features.copy()
display(df_stats.columns)
df_stats = df_stats[['study_id', 'median_steps', 'median_sedentary_time_h', 'median_movement_time_h', 'median_ratio', 'median_steps_2h', 'median_steps_4h', 'median_steps_onset_2h', 'median_steps_onset_4h']]


#calculate mean, median, standard deviation, min, max
df_stats = df_stats.describe().transpose().reset_index()
display(df_stats)
date = datetime.now().strftime("%Y-%m-%d")
df_stats.to_csv(f'../../output/2_descriptive_stats/df_features_step_stats_{date}.csv', index=False)

### Q-Q Plot & Shapiro-Wilk Test   

In [ ]:
step_feature_cols = ["median_steps", "median_sedentary_time_h", "median_movement_time_h", "median_ratio", "median_steps_2h", "median_steps_4h", "median_steps_onset_2h", "median_steps_onset_4h"]
step_summary = check_normality_batch(df_step_features, step_feature_cols, ncols=8)
print(step_summary, f"\n")
date = datetime.now().strftime("%Y-%m-%d")
step_summary.to_csv(f"../../output/2_descriptive_stats/step_normality_summary_{date}.csv", index=False)